In [1]:
import pandas as pd

df = pd.read_csv("../data/telco_churn.csv")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop("customerID", axis=1)
print(df.shape)

(7043, 20)


#### One-hot Encoding

In [2]:
df_encoded = pd.get_dummies(df,  drop_first=True)

print(df_encoded.shape)

(7043, 31)


In [3]:
print(df_encoded.columns.tolist())

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'Churn_Yes']


#### Create X and y and Train-Test Split

In [4]:
X = df_encoded.drop("Churn_Yes", axis=1)
y = df_encoded["Churn_Yes"]

print(X.shape)
print(y.shape)

(7043, 30)
(7043,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(5634, 30)
(1409, 30)
Churn_Yes
False    4139
True     1495
Name: count, dtype: int64
Churn_Yes
False    1035
True      374
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier

In [9]:
# model = RandomForestClassifier(
#     n_estimators=100,
#     random_state=42
# )
#
# model.fit(X_train, y_train)
#
# y_pred = model.predict(X_test)

In [7]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [8]:
print(y_pred[:20])

[False  True False False False  True False False False  True False False
 False  True False False False  True False False]


In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

Accuracy: 0.7693399574166075
[[851 184]
 [141 233]]


In [10]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Precision: 0.5587529976019184
Recall: 0.6229946524064172
F1 Score: 0.5891276864728192


In [11]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(15))

                           Feature  Importance
3                     TotalCharges    0.175931
1                           tenure    0.168000
2                   MonthlyCharges    0.154031
25               Contract_Two year    0.056126
10     InternetService_Fiber optic    0.044604
28  PaymentMethod_Electronic check    0.036394
13              OnlineSecurity_Yes    0.030921
24               Contract_One year    0.029517
4                      gender_Male    0.025737
26            PaperlessBilling_Yes    0.023187
19                 TechSupport_Yes    0.022832
5                      Partner_Yes    0.020787
15                OnlineBackup_Yes    0.020654
6                   Dependents_Yes    0.018957
17            DeviceProtection_Yes    0.018289
